# 12 · SC-FEGAN Pretrained Inference (검은 마스크 → 변형된 얼굴)

**목적**: SC-FEGAN의 pretrained ckpt로 실제 inpainting 결과 생성.

**파이프라인**:
```
사진 → 랜드마크 → compose 9채널 입력 → SC-FEGAN inference → Refinement → 최종
```

**환경**: Colab T4 + **TF 2.x compat mode** (Python 3.11에서 TF 1.15 미지원, compat 사용)

**산출물**: `/MyDrive/cv-final-ckpts/samples/inference_{procedure}.png` 시술별 결과

## 1. 환경 셋업 + SC-FEGAN repo clone

In [ ]:
import os, sys, subprocess
IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    
    # cv-final clone
    REPO = '/content/cv-final'
    if not os.path.exists(REPO):
        from google.colab import userdata
        token = userdata.get('GH_TOKEN').strip()
        result = subprocess.run(
            ['git', 'clone', f'https://{token}@github.com/keonU206/cv-final.git', REPO],
            capture_output=True, text=True,
        )
        if result.returncode != 0:
            raise RuntimeError(f'cv-final clone 실패: {result.stderr}')
    os.chdir(REPO)
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    
    # SC-FEGAN repo clone
    SCFEGAN_REPO = '/content/SC-FEGAN'
    if not os.path.exists(SCFEGAN_REPO):
        !git clone https://github.com/run-youngjoo/SC-FEGAN.git $SCFEGAN_REPO
    
    # TF compat mode
    os.environ['TF_USE_LEGACY_KERAS'] = '1'
    !pip install -q mediapipe==0.10.21 opencv-python pyyaml 'tensorflow<2.16' tf-keras segmentation_models_pytorch

print(f'CWD: {os.getcwd()}')
print(f'SC-FEGAN repo: {SCFEGAN_REPO}')

## 2. SC-FEGAN repo 구조 탐색 (graph API 파악)

SC-FEGAN의 모델 클래스 + 입력 placeholder 이름을 찾아야 함.

In [ ]:
# SC-FEGAN repo 폴더 구조
print('=== SC-FEGAN repo 폴더 구조 ===')
for root, dirs, files in os.walk(SCFEGAN_REPO):
    if any(skip in root for skip in ['__pycache__', '.git', 'imgs']):
        continue
    level = root.replace(SCFEGAN_REPO, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    for f in files:
        if f.endswith('.py'):
            print(f'{indent}  {f}')

# 핵심 .py 파일 첫 30줄 출력 (model 정의 찾기)
print('\n=== 핵심 모델 파일 ===')
for candidate in ['ui/model.py', 'ui/ui.py', 'src/model.py', 'model.py', 'demo.py', 'main.py']:
    path = os.path.join(SCFEGAN_REPO, candidate)
    if os.path.exists(path):
        print(f'\n--- {candidate} (첫 40줄) ---')
        with open(path) as f:
            for i, line in enumerate(f, 1):
                if i > 40: break
                print(f'{i:3}: {line.rstrip()}')

## 3. TF compat 환경 활성화 + SC-FEGAN 모델 빌드

SC-FEGAN repo의 model을 import하여 graph 빌드. 실패 시 다른 import path 시도.

In [ ]:
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # 경고 줄이기

import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()
print(f'TF version: {tf.__version__}')

# SC-FEGAN repo path 추가
import sys
if SCFEGAN_REPO not in sys.path:
    sys.path.insert(0, SCFEGAN_REPO)

# 다양한 import path 시도
Model = None
for import_path in [
    'ui.model',
    'ui.ui',
    'src.model',
    'model',
]:
    try:
        module = __import__(import_path, fromlist=['Model'])
        Model = getattr(module, 'Model', None) or getattr(module, 'SCFEGAN', None)
        if Model:
            print(f'✅ Model 클래스 발견: {import_path}.{Model.__name__}')
            break
    except Exception as e:
        print(f'  {import_path}: {type(e).__name__}: {e}')

if Model is None:
    print('\n⚠ Model 클래스 자동 발견 실패. 셀 2 출력 보고 수동 import 필요.')

## 4. 사진 + 랜드마크 추출

In [ ]:
import numpy as np, cv2, matplotlib.pyplot as plt
from pathlib import Path

SIZE = 512
OUTPUT_DIR = Path('outputs/scfegan_inference')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if IS_COLAB:
    from google.colab import files
    print('얼굴 사진 1장 업로드:')
    uploaded = files.upload()
    IMG_PATH = list(uploaded.keys())[0]
else:
    IMG_PATH = 'demo_face.jpg'

image_rgb = cv2.cvtColor(cv2.imread(IMG_PATH), cv2.COLOR_BGR2RGB)
image_rgb = cv2.resize(image_rgb, (SIZE, SIZE))

import mediapipe as mp
with mp.solutions.face_mesh.FaceMesh(
    static_image_mode=True, max_num_faces=1, refine_landmarks=True,
) as fm:
    res = fm.process(image_rgb)

if not res.multi_face_landmarks:
    raise RuntimeError('얼굴 감지 실패')

lms = res.multi_face_landmarks[0].landmark
landmarks = np.array([[lm.x * SIZE, lm.y * SIZE] for lm in lms], dtype=np.int32)
print(f'✅ Landmarks: {landmarks.shape}')

plt.figure(figsize=(5, 5))
plt.imshow(image_rgb); plt.axis('off'); plt.title('Input')
plt.show()

## 5. 9채널 입력 생성

In [ ]:
from project.input_generator import compose_scfegan_input, to_scfegan_tensor

PROCEDURES = ['nose_tip', 'double_eyelid', 'jaw_v_line']
composed_all = {}

for proc_id in PROCEDURES:
    composed = compose_scfegan_input(
        image_rgb, landmarks, proc_id,
        size=SIZE, intensity=0.6, seed=42,
    )
    composed_all[proc_id] = composed
    
    # 9채널 tensor 변환
    tensor = to_scfegan_tensor(
        composed, normalize=True, channels_first=False, batch_dim=True,
    )
    print(f'{proc_id}: tensor shape {tensor.shape}, range [{tensor.min():.2f}, {tensor.max():.2f}]')

## 6. SC-FEGAN Inference

⚠ 이 셀은 셀 2/3에서 발견한 Model 클래스에 따라 수정 필요할 수 있음.

SC-FEGAN 원본 graph 변수 이름 (run-youngjoo 코드 기준):
- 입력 placeholder: 보통 `input_ph` 또는 `inputs`
- 출력 tensor: 보통 `gen_out`, `output`, 또는 `G_out`

In [ ]:
CKPT_PATH = '/content/drive/MyDrive/SC-FEGAN-ckpt/SC-FEGAN.ckpt'
print(f'ckpt 존재: {os.path.exists(CKPT_PATH + ".index")}')

# ckpt 변수 이름 확인 (graph 구조 추론용)
from tensorflow.python import pywrap_tensorflow
reader = pywrap_tensorflow.NewCheckpointReader(CKPT_PATH)
shape_map = reader.get_variable_to_shape_map()
print(f'\n=== ckpt 변수 개수: {len(shape_map)} ===')
print('첫 20개 변수 이름:')
for k in sorted(shape_map.keys())[:20]:
    print(f'  {k}: {shape_map[k]}')

In [ ]:
# Model 클래스 빌드 + ckpt 로드
# ⚠ 셀 2/3 출력 보고 model 빌드 방식 결정

tf.reset_default_graph()

try:
    if Model is not None:
        # 자동 발견된 Model 클래스 사용
        model = Model(
            mode='test' if hasattr(Model, 'mode') else None,
        )
        print('✅ Model 인스턴스 생성')
        print(f'  attributes: {[a for a in dir(model) if not a.startswith("_")]}')
    else:
        raise RuntimeError('Model 클래스 없음 — 수동 빌드 필요')
    
    saver = tf.train.Saver()
    config = tf.ConfigProto(allow_soft_placement=True)
    config.gpu_options.allow_growth = True
    sess = tf.Session(config=config)
    saver.restore(sess, CKPT_PATH)
    print(f'✅ ckpt 로드 완료: {CKPT_PATH}')
    
except Exception as e:
    print(f'❌ Model 빌드 실패: {e}')
    import traceback
    traceback.print_exc()
    print('\n→ 셀 2/3 출력 확인 후 수동 import + graph 빌드 필요')

In [ ]:
# Inference 실행 — 시술별 결과 생성
# ⚠ Model 클래스의 정확한 placeholder/output 이름은 셀 2/3에서 확인

results = {}

def run_inference(composed_dict):
    """SC-FEGAN inference 실행 — Model API 확정 후 호출."""
    tensor = to_scfegan_tensor(
        composed_dict, normalize=True, channels_first=False, batch_dim=True,
    )
    
    # 가능한 placeholder/output 이름 시도
    placeholder_candidates = ['input_ph', 'inputs', 'images', 'X']
    output_candidates = ['gen_out', 'output', 'G_out', 'generated', 'fake']
    
    input_ph = None
    output_op = None
    for name in placeholder_candidates:
        if hasattr(model, name):
            input_ph = getattr(model, name)
            print(f'  input placeholder: model.{name}')
            break
    for name in output_candidates:
        if hasattr(model, name):
            output_op = getattr(model, name)
            print(f'  output tensor: model.{name}')
            break
    
    if input_ph is None or output_op is None:
        raise RuntimeError(
            f'placeholder/output 자동 발견 실패. '
            f'model.attributes 확인 후 수동 지정 필요: '
            f'{[a for a in dir(model) if not a.startswith("_")]}'
        )
    
    output = sess.run(output_op, feed_dict={input_ph: tensor})
    return ((output[0] + 1) * 127.5).clip(0, 255).astype(np.uint8)

for proc_id, composed in composed_all.items():
    print(f'\n{proc_id} inference...')
    try:
        result = run_inference(composed)
        results[proc_id] = result
        print(f'  ✅ shape: {result.shape}')
    except Exception as e:
        print(f'  ❌ {type(e).__name__}: {e}')
        results[proc_id] = composed['incomplete_image']  # fallback

## 7. 결과 시각화 + Refinement 적용

In [ ]:
# Refinement 모델 로드
import torch
from project.refinement.model import build_refinement_wrapper

device = 'cuda' if torch.cuda.is_available() else 'cpu'
REF_CKPT = '/content/drive/MyDrive/cv-final-ckpts/refinement_best.pt'

if os.path.exists(REF_CKPT):
    ref_model = build_refinement_wrapper(encoder_weights=None).to(device)
    ref_model.load_state_dict(torch.load(REF_CKPT, map_location=device))
    ref_model.eval()
    print(f'✅ Refinement 모델 로드: {REF_CKPT}')
else:
    ref_model = None
    print(f'⚠ Refinement ckpt 없음 → 적용 안 함')

def refine(img_rgb):
    if ref_model is None:
        return img_rgb
    tensor = torch.from_numpy(img_rgb).float() / 127.5 - 1.0
    tensor = tensor.permute(2, 0, 1).unsqueeze(0).to(device)
    with torch.no_grad():
        out = ref_model(tensor)[0].cpu()
    return ((out.clamp(-1, 1) + 1) * 127.5).byte().permute(1, 2, 0).numpy()

In [ ]:
# 시술 3종 × 4컬럼 (Before / SC-FEGAN out / Refinement / Mask)
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for i, proc_id in enumerate(PROCEDURES):
    after_scfegan = results[proc_id]
    after_refined = refine(after_scfegan)
    mask = composed_all[proc_id]['mask'][..., 0]
    
    axes[i, 0].imshow(image_rgb)
    axes[i, 0].set_title(f'{proc_id} · Before')
    axes[i, 1].imshow(after_scfegan)
    axes[i, 1].set_title('SC-FEGAN 출력')
    axes[i, 2].imshow(after_refined)
    axes[i, 2].set_title('+ Refinement (최종)')
    axes[i, 3].imshow(mask, cmap='gray')
    axes[i, 3].set_title('Mask 영역')
    for ax in axes[i]:
        ax.axis('off')
    
    # 개별 저장
    cv2.imwrite(str(OUTPUT_DIR / f'final_{proc_id}.png'),
                cv2.cvtColor(after_refined, cv2.COLOR_RGB2BGR))

plt.tight_layout()
save_path = OUTPUT_DIR / 'all_results.png'
plt.savefig(save_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'\n✅ 저장: {save_path}')

## 8. Drive 백업

In [ ]:
if IS_COLAB:
    DRIVE_OUT = Path('/content/drive/MyDrive/cv-final-ckpts/samples')
    DRIVE_OUT.mkdir(parents=True, exist_ok=True)
    !cp -r {OUTPUT_DIR}/* {DRIVE_OUT}/
    print(f'✅ Drive 백업: {DRIVE_OUT}/')
    !ls -lh {DRIVE_OUT}/

print('\n=== Phase 7-E 완료 ===')
print('이 결과를 발표 슬라이드에 사용하거나, Streamlit에서 정적 이미지로 표시 가능.')